# Silver to Gold Transformation
## Pipeline Monitor — Databricks Notebook 02

This notebook reads clean Delta tables from Silver zone,
joins orders with customers, calculates LTV scores using RFM model,
and builds 5 Gold tables ready for business analytics.

### Gold Tables Built:
1. **dim_customers** — Clean customer dimension
2. **fct_orders** — Orders enriched with customer segment
3. **mart_segment_revenue** — Revenue by segment + country
4. **mart_customer_ltv** — RFM Lifetime Value scores
5. **mart_customer_activity** — 90-day activity summary

In [0]:
# Cell 1 — Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime

STORAGE_ACCOUNT = dbutils.secrets.get(scope="pipeline-monitor", key="adls-account-name")
STORAGE_KEY     = dbutils.secrets.get(scope="pipeline-monitor", key="adls-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    STORAGE_KEY
)

BRONZE = f"wasbs://bronze@{STORAGE_ACCOUNT}.blob.core.windows.net"
SILVER = f"wasbs://silver@{STORAGE_ACCOUNT}.blob.core.windows.net"
GOLD   = f"wasbs://gold@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"✅ Storage configured: {STORAGE_ACCOUNT}")
print(f"✅ Silver path: {SILVER}")
print(f"✅ Gold path: {GOLD}")

## Step 1: Read Silver Tables

Reads all 4 clean Delta tables from Silver zone.
These are the inputs for all Gold table calculations.

In [0]:
# Cell 2 — Read all Silver tables
print("Reading Silver tables...")

orders    = spark.read.format("delta").load(f"{SILVER}/orders_cleansed/")
products  = spark.read.format("delta").load(f"{SILVER}/products_cleansed/")
customers = spark.read.format("delta").load(f"{SILVER}/customers_cleansed/")
events    = spark.read.format("delta").load(f"{SILVER}/customer_events_cleansed/")

print(f"  ✅ Orders:    {orders.count()} rows")
print(f"  ✅ Products:  {products.count()} rows")
print(f"  ✅ Customers: {customers.count()} rows")
print(f"  ✅ Events:    {events.count()} rows")

## Step 2: Build dim_customers

Clean customer dimension table.
One row per customer with all profile information.
Used as a lookup table in all other Gold tables.

In [0]:
# Cell 3 — Build dim_customers
print("\n[1/5] Building dim_customers...")

dim_customers = (
    customers
    .withColumn("customer_age_days",
        F.datediff(F.current_date(), F.col("signup_date")))
    .select(
        "customer_id", "full_name", "email", "country", "city",
        "segment", "acquisition_channel", "signup_date",
        "total_orders", "total_spend", "avg_order_value",
        "last_order_date", "days_since_last_order",
        "is_email_verified", "preferred_currency", "customer_age_days"
    )
)

(dim_customers.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .save(f"{GOLD}/dim_customers/"))

print(f"  ✅ dim_customers: {dim_customers.count()} rows written to Gold")

## Step 3: Build fct_orders

Core fact table — orders enriched with customer segment
and product category. One row per order.

This is the most important table — it links orders to customers
and enables revenue analysis by segment, country, and channel.

In [0]:
# Cell 4 — Build fct_orders
print("\n[2/5] Building fct_orders...")

from delta.tables import DeltaTable

fct_orders = (
    orders
    .join(customers.select(
              "customer_id", "full_name", "country", "city",
              "segment", "acquisition_channel", "signup_date"),
          on="customer_id", how="left")
    .join(products.select("product_id", "category", "price"),
          on="product_id", how="left")
    .withColumn("order_date",       F.to_date("order_timestamp"))
    .withColumn("order_hour",       F.hour("order_timestamp"))
    .withColumn("customer_segment", F.col("segment"))
    .withColumn("product_category", F.col("category"))
    .select(
        "order_id", "customer_id", "full_name", "country", "city",
        "customer_segment", "acquisition_channel",
        "order_timestamp", "order_date", "order_hour",
        "status", "amount", "currency", "source_channel",
        "product_id", "product_category", "ingested_at"
    )
)

# Delta MERGE for upsert — prevents duplicate rows on re-runs
if DeltaTable.isDeltaTable(spark, f"{GOLD}/fct_orders/"):
    dt = DeltaTable.forPath(spark, f"{GOLD}/fct_orders/")
    (dt.alias("old")
       .merge(fct_orders.alias("new"), "old.order_id = new.order_id")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute())
    print("  ✅ fct_orders: MERGE upsert complete")
else:
    fct_orders.write.format("delta").mode("overwrite").save(f"{GOLD}/fct_orders/")
    print(f"  ✅ fct_orders: initial write — {fct_orders.count()} rows")

## Step 4: Build mart_segment_revenue

Daily revenue breakdown by customer segment and country.
This is the table a CMO or Head of Growth would use to
track which customer segments are driving revenue.

Aggregations:
- Total orders and revenue per day
- Average order value
- Unique customers
- Revenue per customer

In [0]:
# Cell 5 — Build mart_segment_revenue
print("\n[3/5] Building mart_segment_revenue...")

mart_segment = (
    fct_orders
    .filter(F.col("status") == "completed")
    .groupBy("order_date", "customer_segment", "country")
    .agg(
        F.count("order_id")            .alias("total_orders"),
        F.sum("amount")                .alias("total_revenue"),
        F.avg("amount")                .alias("avg_order_value"),
        F.countDistinct("customer_id") .alias("unique_customers"),
    )
    .withColumn("total_revenue",     F.round("total_revenue", 2))
    .withColumn("avg_order_value",   F.round("avg_order_value", 2))
    .withColumn("revenue_per_customer",
        F.round(F.col("total_revenue") / F.col("unique_customers"), 2))
)

mart_segment.write.format("delta").mode("overwrite") \
            .save(f"{GOLD}/mart_segment_revenue/")

print(f"  ✅ mart_segment_revenue: {mart_segment.count()} rows written to Gold")

## Step 5: Build mart_customer_ltv

Lifetime Value scoring using RFM model.
Every customer gets scored 1-5 on three dimensions:

- **Recency (R)** — days since last order (lower = better)
- **Frequency (F)** — total number of orders (higher = better)  
- **Monetary (M)** — total spend (higher = better)

Final LTV score = M*0.5 + F*0.3 + R*0.2

LTV Tiers:
- **Champions** → score >= 4.0
- **Loyal** → score >= 3.0
- **At Risk** → score >= 2.0
- **Lost** → score < 2.0

In [0]:
# Cell 6 — Build mart_customer_ltv (RFM Model)
print("\n[4/5] Building mart_customer_ltv (RFM model)...")

# Step 1: Aggregate orders per customer
customer_orders = (
    fct_orders
    .groupBy("customer_id")
    .agg(
        F.max("order_date")  .alias("last_order_date"),
        F.min("order_date")  .alias("first_order_date"),
        F.count("order_id")  .alias("order_count"),
        F.sum("amount")      .alias("total_revenue"),
        F.avg("amount")      .alias("avg_order_value"),
    )
    .withColumn("recency_days",
        F.datediff(F.current_date(), F.col("last_order_date")))
    .withColumn("total_revenue",   F.round("total_revenue", 2))
    .withColumn("avg_order_value", F.round("avg_order_value", 2))
)

# Step 2: Calculate percentile buckets for scoring
r_q = customer_orders.approxQuantile("recency_days",  [0.2,0.4,0.6,0.8], 0.01)
f_q = customer_orders.approxQuantile("order_count",   [0.2,0.4,0.6,0.8], 0.01)
m_q = customer_orders.approxQuantile("total_revenue", [0.2,0.4,0.6,0.8], 0.01)

def score(col, q, ascending=True):
    """Score a column 1-5 based on percentile buckets."""
    if ascending:  # lower is better (recency)
        return (F.when(F.col(col) <= q[0], 5)
                 .when(F.col(col) <= q[1], 4)
                 .when(F.col(col) <= q[2], 3)
                 .when(F.col(col) <= q[3], 2)
                 .otherwise(1))
    else:          # higher is better (frequency, monetary)
        return (F.when(F.col(col) >= q[3], 5)
                 .when(F.col(col) >= q[2], 4)
                 .when(F.col(col) >= q[1], 3)
                 .when(F.col(col) >= q[0], 2)
                 .otherwise(1))

# Step 3: Build LTV mart with scores and tiers
mart_ltv = (
    customer_orders
    .join(dim_customers.select(
              "customer_id", "full_name", "segment",
              "country", "acquisition_channel", "signup_date"),
          on="customer_id", how="left")
    .withColumn("r_score", score("recency_days",  r_q, ascending=True))
    .withColumn("f_score", score("order_count",   f_q, ascending=False))
    .withColumn("m_score", score("total_revenue", m_q, ascending=False))
    .withColumn("ltv_score",
        F.round(F.col("m_score")*0.5 +
                F.col("f_score")*0.3 +
                F.col("r_score")*0.2, 2))
    .withColumn("ltv_tier",
        F.when(F.col("ltv_score") >= 4.0, "Champions")
         .when(F.col("ltv_score") >= 3.0, "Loyal")
         .when(F.col("ltv_score") >= 2.0, "At Risk")
         .otherwise("Lost"))
    .select(
        "customer_id", "full_name", "segment", "country",
        "acquisition_channel", "signup_date",
        "last_order_date", "first_order_date",
        "recency_days", "order_count", "total_revenue",
        "avg_order_value", "r_score", "f_score", "m_score",
        "ltv_score", "ltv_tier"
    )
)

mart_ltv.write.format("delta").mode("overwrite") \
        .save(f"{GOLD}/mart_customer_ltv/")

print(f"  ✅ mart_customer_ltv: {mart_ltv.count()} customers scored")

# Show LTV tier distribution
print("\n  LTV Tier Distribution:")
mart_ltv.groupBy("ltv_tier").count().orderBy("count", ascending=False).show()

## Step 6: Build mart_customer_activity

90-day rolling activity summary per customer.
Combines order data with behavioural events to show
how engaged each customer is with the platform.

Activity Tiers:
- **Highly Active** → 5+ orders in last 90 days
- **Active** → 2-4 orders in last 90 days
- **Low Activity** → 1 order in last 90 days
- **Inactive** → no orders in last 90 days

Event signals tracked:
- Logins, support tickets, return requests

In [0]:
# Cell 7 — Build mart_customer_activity
print("\n[5/5] Building mart_customer_activity...")

# Orders in last 90 days per customer
orders_90d = (
    fct_orders
    .filter(F.datediff(F.current_date(), F.col("order_date")) <= 90)
    .groupBy("customer_id")
    .agg(
        F.count("order_id")  .alias("orders_last_90d"),
        F.sum("amount")      .alias("revenue_last_90d"),
        F.avg("amount")      .alias("avg_order_value_last_90d"),
    )
    .withColumn("revenue_last_90d",         F.round("revenue_last_90d", 2))
    .withColumn("avg_order_value_last_90d", F.round("avg_order_value_last_90d", 2))
)

# Events in last 90 days per customer
events_90d = (
    events
    .filter(F.datediff(F.current_date(), F.to_date("event_ts")) <= 90)
    .groupBy("customer_id")
    .agg(
        F.count("event_id").alias("total_events_last_90d"),
        F.sum(F.when(F.col("event_type") == "login",            1).otherwise(0))
          .alias("logins_last_90d"),
        F.sum(F.when(F.col("event_type") == "support_ticket",   1).otherwise(0))
          .alias("support_tickets_last_90d"),
        F.sum(F.when(F.col("event_type") == "return_request",   1).otherwise(0))
          .alias("returns_last_90d"),
    )
)

# Join everything together
mart_activity = (
    dim_customers.select(
        "customer_id", "full_name", "segment",
        "country", "days_since_last_order")
    .join(orders_90d, on="customer_id", how="left")
    .join(events_90d, on="customer_id", how="left")
    .fillna(0, subset=[
        "orders_last_90d", "revenue_last_90d",
        "avg_order_value_last_90d", "total_events_last_90d",
        "logins_last_90d", "support_tickets_last_90d", "returns_last_90d"
    ])
    .withColumn("activity_tier",
        F.when(F.col("orders_last_90d") >= 5, "Highly Active")
         .when(F.col("orders_last_90d") >= 2, "Active")
         .when(F.col("orders_last_90d") >= 1, "Low Activity")
         .otherwise("Inactive"))
)

mart_activity.write.format("delta").mode("overwrite") \
             .save(f"{GOLD}/mart_customer_activity/")

print(f"  ✅ mart_customer_activity: {mart_activity.count()} rows written to Gold")

# Show activity tier distribution
print("\n  Activity Tier Distribution:")
mart_activity.groupBy("activity_tier").count().orderBy("count", ascending=False).show()

## ✅ Silver → Gold Complete

All 5 Gold tables successfully built:
- dim_customers — 300 customer profiles
- fct_orders — 100 enriched orders
- mart_segment_revenue — revenue by segment + country
- mart_customer_ltv — RFM scored customers
- mart_customer_activity — 90-day activity summary

Next: Load Gold tables into Azure SQL (pipeline-gold database)

In [0]:
# Cell 8 — Verify all Gold tables
print("\n=== Gold Zone Verification ===")

gold_tables = [
    "dim_customers",
    "fct_orders",
    "mart_segment_revenue",
    "mart_customer_ltv",
    "mart_customer_activity",
]

for table in gold_tables:
    df = spark.read.format("delta").load(f"{GOLD}/{table}/")
    print(f"  ✅ {table}: {df.count()} rows | {len(df.columns)} columns")

print("\n=== All Gold tables verified ===")
dbutils.notebook.exit("SUCCESS")